In [1]:
import cv2
import numpy as np
import tensorflow as tf
from collections import deque
import time
from tensorflow.keras.models import load_model

In [2]:
# Model paths
ACTION_MODEL_PATH = "action_cnn_lstm.keras"  # Your trained action model
EMOTION_MODEL_PATH = "fine_tuned_best.keras"  # Your trained emotion model


In [ ]:
# Class labels
ACTION_CLASSES = [
    "WalkingWithDog",  # Walking with a dog
    "PushUps",         
    "BrushingTeeth",   
    "BlowingCandles",  
    "Typing",          
    "Archery",         # Archery
    "Basketball",      
    "Bowling",         #
    "BoxingPunchingBag", 
    "HorseRace"        #
]
EMOTION_CLASSES = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']


In [ ]:
9     # Video parameters
ACTION_SEQ_LENGTH = 16      # Number of frames for action recognition
ACTION_IMG_SIZE = (64, 64)  # Action model input size (height, width)
EMOTION_IMG_SIZE = (48, 48) # Emotion model input size (height, width)


In [5]:
# Face detection
FACE_CASCADE_PATH = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'


In [6]:
# Display settings
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.7
FONT_THICKNESS = 2
TEXT_COLOR = (0, 255, 0)
BOX_COLOR = (255, 0, 0)

In [7]:
# ============================================
# LOAD MODELS
# ============================================
def load_models(action_path, emotion_path):
    """Load both trained models"""
    print("Loading models...")
    
    try:
        action_model = load_model(action_path)
        print(f"✅ Action model loaded from: {action_path}")
    except Exception as e:
        print(f"❌ Error loading action model: {e}")
        action_model = None
    
    try:
        emotion_model = load_model(emotion_path)
        print(f"✅ Emotion model loaded from: {emotion_path}")
    except Exception as e:
        print(f"❌ Error loading emotion model: {e}")
        emotion_model = None
    
    return action_model, emotion_model

In [8]:
# ============================================
# PREPROCESSING FUNCTIONS
# ============================================
def preprocess_frame_for_action(frame, target_size=ACTION_IMG_SIZE):
    """Preprocess frame for action recognition"""
    # Resize to action model input size
    resized = cv2.resize(frame, target_size)
    # Convert to RGB
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    # Normalize
    normalized = rgb.astype("float32") / 255.0
    return normalized

def preprocess_face_for_emotion(face_img, target_size=EMOTION_IMG_SIZE):
    """Preprocess face for emotion recognition with CLAHE"""
    # Convert to grayscale
    gray = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)
    # Resize
    resized = cv2.resize(gray, target_size)
    # Apply CLAHE (matches training preprocessing)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(resized)
    # Normalize
    normalized = enhanced.astype("float32") / 255.0
    # Add channel dimension
    normalized = np.expand_dims(normalized, axis=-1)
    return normalized

In [9]:
# ============================================
# FACE DETECTION
# ============================================
def detect_faces(frame, face_cascade):
    """Detect faces in frame using Haar Cascade"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )
    return faces

In [10]:
def predict_action(action_model, frame_sequence):
    """Predict action from sequence of frames"""
    if len(frame_sequence) < ACTION_SEQ_LENGTH:
        return None, 0.0
    
    # Prepare sequence
    sequence = np.array(list(frame_sequence))
    sequence = np.expand_dims(sequence, axis=0)  # Add batch dimension
    
    # Predict
    predictions = action_model.predict(sequence, verbose=0)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class]
    
    return ACTION_CLASSES[predicted_class], confidence

In [11]:
def predict_emotion(emotion_model, face_img):
    """Predict emotion from face image"""
    if face_img is None or face_img.size == 0:
        return None, 0.0
    
    # Preprocess
    processed_face = preprocess_face_for_emotion(face_img)
    processed_face = np.expand_dims(processed_face, axis=0)  # Add batch dimension
    
    # Predict
    predictions = emotion_model.predict(processed_face, verbose=0)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class]
    
    return EMOTION_CLASSES[predicted_class], confidence

In [12]:
class FPSCalculator:
    """Calculate real-time FPS and latency"""
    def __init__(self, buffer_size=30):
        self.frame_times = deque(maxlen=buffer_size)
        self.processing_times = deque(maxlen=buffer_size)
    
    def update(self, processing_time):
        """Update with new frame timing"""
        current_time = time.time()
        self.frame_times.append(current_time)
        self.processing_times.append(processing_time)
    
    def get_fps(self):
        """Calculate current FPS"""
        if len(self.frame_times) < 2:
            return 0.0
        time_diff = self.frame_times[-1] - self.frame_times[0]
        if time_diff > 0:
            return (len(self.frame_times) - 1) / time_diff
        return 0.0
    
    def get_avg_latency(self):
        """Calculate average processing latency (ms)"""
        if len(self.processing_times) == 0:
            return 0.0
        return np.mean(self.processing_times) * 1000  # Convert to ms


In [13]:
def draw_info_panel(frame, fps, latency, action, action_conf, emotions_data):
    """Draw information panel on frame"""
    height, width = frame.shape[:2]
    panel_height = 150
    
    # Create semi-transparent panel
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (width, panel_height), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    
    # FPS and Latency
    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30),
                FONT, FONT_SCALE, (0, 255, 255), FONT_THICKNESS)
    cv2.putText(frame, f"Latency: {latency:.1f}ms", (10, 60),
                FONT, FONT_SCALE, (0, 255, 255), FONT_THICKNESS)
    
    # Action prediction
    if action:
        action_text = f"Action: {action} ({action_conf*100:.1f}%)"
        cv2.putText(frame, action_text, (10, 90),
                    FONT, FONT_SCALE, (0, 255, 0), FONT_THICKNESS)
    
    # Emotion count
    cv2.putText(frame, f"Faces: {len(emotions_data)}", (10, 120),
                FONT, FONT_SCALE, (255, 255, 0), FONT_THICKNESS)

In [14]:
def draw_face_emotion(frame, face_rect, emotion, confidence):
    """Draw bounding box and emotion label on face"""
    x, y, w, h = face_rect
    
    # Draw rectangle around face
    cv2.rectangle(frame, (x, y), (x+w, y+h), BOX_COLOR, 2)
    
    # Prepare emotion text
    emotion_text = f"{emotion}: {confidence*100:.1f}%"
    
    # Get text size for background
    (text_width, text_height), baseline = cv2.getTextSize(
        emotion_text, FONT, FONT_SCALE, FONT_THICKNESS
    )
    
    # Draw text background
    cv2.rectangle(frame, 
                  (x, y - text_height - 10),
                  (x + text_width, y),
                  BOX_COLOR, -1)
    
    # Draw emotion text
    cv2.putText(frame, emotion_text, (x, y - 5),
                FONT, FONT_SCALE, (255, 255, 255), FONT_THICKNESS)

In [ ]:
def run_realtime_system(action_model, emotion_model, camera_id=0):
    """Main function for real-time action and emotion recognition"""
    
    print("\n" + "="*60)
    print("STARTING REAL-TIME RECOGNITION SYSTEM")
    print("="*60)
    print("Controls:")
    print("  - Press 'q' to quit")
    print("  - Press 's' to save current frame")
    print("  - Press 'p' to pause/resume")
    print("="*60 + "\n")
    
    # Initialize video capture
    cap = cv2.VideoCapture(camera_id)
    
    if not cap.isOpened():
        print("❌ Error: Could not open camera")
        return
    
    # Set camera properties for better performance
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    # Initialize face detector
    face_cascade = cv2.CascadeClassifier(FACE_CASCADE_PATH)
    
    # Initialize frame buffer for action recognition
    frame_buffer = deque(maxlen=ACTION_SEQ_LENGTH)
    
    # Initialize FPS calculator
    fps_calc = FPSCalculator()
    
    # State variables
    current_action = None
    current_action_conf = 0.0
    paused = False
    frame_count = 0
    
    print("✅ System started. Processing frames...\n")
    
    try:
        while True:
            if not paused:
                # Start timing
                start_time = time.time()
                
                # Capture frame
                ret, frame = cap.read()
                if not ret:
                    print("❌ Error: Could not read frame")
                    break
                
                frame_count += 1
                
                # Create a copy for display
                display_frame = frame.copy()
                
                # ============================================
                # ACTION RECOGNITION
                # ============================================
                if action_model is not None:
                    # Preprocess and add to buffer
                    processed_frame = preprocess_frame_for_action(frame)
                    frame_buffer.append(processed_frame)
                    
                    # Predict action every 8 frames (reduce computation)
                    if frame_count % 8 == 0 and len(frame_buffer) == ACTION_SEQ_LENGTH:
                        current_action, current_action_conf = predict_action(
                            action_model, frame_buffer
                        )
                
                # ============================================
                # EMOTION RECOGNITION
                # ============================================
                emotions_data = []
                if emotion_model is not None:
                    # Detect faces
                    faces = detect_faces(frame, face_cascade)
                    
                    # Process each detected face
                    for (x, y, w, h) in faces:
                        # Extract face region
                        face_img = frame[y:y+h, x:x+w]
                        
                        # Predict emotion
                        emotion, confidence = predict_emotion(emotion_model, face_img)
                        
                        if emotion:
                            emotions_data.append({
                                'rect': (x, y, w, h),
                                'emotion': emotion,
                                'confidence': confidence
                            })
                            
                            # Draw face and emotion
                            draw_face_emotion(display_frame, (x, y, w, h), 
                                            emotion, confidence)
                
                # ============================================
                # VISUALIZATION
                # ============================================
                # Calculate performance metrics
                processing_time = time.time() - start_time
                fps_calc.update(processing_time)
                current_fps = fps_calc.get_fps()
                avg_latency = fps_calc.get_avg_latency()
                
                # Draw info panel
                draw_info_panel(display_frame, current_fps, avg_latency,
                              current_action, current_action_conf, emotions_data)
                
                # Display frame
                cv2.imshow('Real-Time Action & Emotion Recognition', display_frame)
            
            # ============================================
            # KEYBOARD CONTROLS
            # ============================================
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q'):
                print("\n🛑 Quitting...")
                break
            elif key == ord('s'):
                filename = f"capture_{int(time.time())}.jpg"
                cv2.imwrite(filename, display_frame)
                print(f"📸 Saved frame: {filename}")
            elif key == ord('p'):
                paused = not paused
                status = "⏸️  PAUSED" if paused else "▶️  RESUMED"
                print(status)
    
    except KeyboardInterrupt:
        print("\n⚠️ Interrupted by user")
    
    finally:
        # Cleanup
        cap.release()
        cv2.destroyAllWindows()
        
        # Print final statistics
        print("\n" + "="*60)
        print("SESSION STATISTICS")
        print("="*60)
        print(f"Total frames processed: {frame_count}")
        print(f"Average FPS: {fps_calc.get_fps():.1f}")
        print(f"Average latency: {fps_calc.get_avg_latency():.1f}ms")
        print("="*60 + "\n")

In [16]:
def benchmark_models(action_model, emotion_model, num_iterations=100):
    """Benchmark model inference times"""
    
    print("\n" + "="*60)
    print("MODEL PERFORMANCE BENCHMARK")
    print("="*60 + "\n")
    
    # Create dummy inputs
    action_input = np.random.rand(1, ACTION_SEQ_LENGTH, *ACTION_IMG_SIZE, 3).astype(np.float32)
    emotion_input = np.random.rand(1, *EMOTION_IMG_SIZE, 1).astype(np.float32)
    
    # Benchmark action model
    if action_model is not None:
        print("Benchmarking Action Model...")
        action_times = []
        for _ in range(num_iterations):
            start = time.time()
            _ = action_model.predict(action_input, verbose=0)
            action_times.append(time.time() - start)
        
        print(f"  Avg inference time: {np.mean(action_times)*1000:.2f}ms")
        print(f"  Min: {np.min(action_times)*1000:.2f}ms")
        print(f"  Max: {np.max(action_times)*1000:.2f}ms")
        print(f"  Throughput: {1/np.mean(action_times):.1f} FPS\n")
    
    # Benchmark emotion model
    if emotion_model is not None:
        print("Benchmarking Emotion Model...")
        emotion_times = []
        for _ in range(num_iterations):
            start = time.time()
            _ = emotion_model.predict(emotion_input, verbose=0)
            emotion_times.append(time.time() - start)
        
        print(f"  Avg inference time: {np.mean(emotion_times)*1000:.2f}ms")
        print(f"  Min: {np.min(emotion_times)*1000:.2f}ms")
        print(f"  Max: {np.max(emotion_times)*1000:.2f}ms")
        print(f"  Throughput: {1/np.mean(emotion_times):.1f} FPS\n")
    
    print("="*60 + "\n")

In [17]:
def process_video_file(action_model, emotion_model, video_path, output_path):
    """Process a video file (alternative to webcam)"""
    
    print(f"\n📹 Processing video: {video_path}")
    
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print("❌ Error: Could not open video file")
        return
    
    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Initialize
    face_cascade = cv2.CascadeClassifier(FACE_CASCADE_PATH)
    frame_buffer = deque(maxlen=ACTION_SEQ_LENGTH)
    fps_calc = FPSCalculator()
    
    current_action = None
    current_action_conf = 0.0
    frame_count = 0
    
    print(f"Video info: {width}x{height} @ {fps} FPS, {total_frames} frames")
    print("Processing...\n")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        start_time = time.time()
        frame_count += 1
        
        display_frame = frame.copy()
        
        # Action recognition
        if action_model is not None:
            processed_frame = preprocess_frame_for_action(frame)
            frame_buffer.append(processed_frame)
            
            if frame_count % 8 == 0 and len(frame_buffer) == ACTION_SEQ_LENGTH:
                current_action, current_action_conf = predict_action(
                    action_model, frame_buffer
                )
        
        # Emotion recognition
        emotions_data = []
        if emotion_model is not None:
            faces = detect_faces(frame, face_cascade)
            
            for (x, y, w, h) in faces:
                face_img = frame[y:y+h, x:x+w]
                emotion, confidence = predict_emotion(emotion_model, face_img)
                
                if emotion:
                    emotions_data.append({
                        'rect': (x, y, w, h),
                        'emotion': emotion,
                        'confidence': confidence
                    })
                    draw_face_emotion(display_frame, (x, y, w, h), 
                                    emotion, confidence)
        
        # Calculate metrics
        processing_time = time.time() - start_time
        fps_calc.update(processing_time)
        current_fps = fps_calc.get_fps()
        avg_latency = fps_calc.get_avg_latency()
        
        # Draw info
        draw_info_panel(display_frame, current_fps, avg_latency,
                       current_action, current_action_conf, emotions_data)
        
        # Write frame
        out.write(display_frame)
        
        # Progress
        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Progress: {progress:.1f}% ({frame_count}/{total_frames})", end='\r')
    
    cap.release()
    out.release()
    
    print(f"\n✅ Processed video saved: {output_path}")
    print(f"Average FPS: {fps_calc.get_fps():.1f}")
    print(f"Average latency: {fps_calc.get_avg_latency():.1f}ms\n")


In [ ]:

def main():
    """Main function to run the real-time system"""
    
    # Load models
    action_model, emotion_model = load_models(ACTION_MODEL_PATH, EMOTION_MODEL_PATH)
    
    if action_model is None and emotion_model is None:
        print("❌ No models loaded. Exiting.")
        return
    
    # Run benchmark
    benchmark_models(action_model, emotion_model, num_iterations=50)
    while True:
        # Choose mode
        print("Select mode:")
        print("1. Real-time webcam")
        print("2. Process video file")
        choice = input("Enter choice (1 or 2): ").strip()
        
        if choice == "1":
            # Real-time webcam
            run_realtime_system(action_model, emotion_model, camera_id=0)
        
        elif choice == "2":
            # Video file processing
            video_path = input("Enter video file path: ").strip()
            output_path = input("Enter output file path (default: output.mp4): ").strip()
            if not output_path:
                output_path = "output.mp4"
            process_video_file(action_model, emotion_model, video_path, output_path)
        
        else:
            print("Invalid choice")
            break

In [ ]:
# ============================================
# RUN
# ============================================
if __name__ == "__main__":
    main()

Loading models...
✅ Action model loaded from: action_cnn_lstm.keras
✅ Emotion model loaded from: fine_tuned_best.keras

MODEL PERFORMANCE BENCHMARK

Benchmarking Action Model...
  Avg inference time: 454.84ms
  Min: 128.00ms
  Max: 15728.07ms
  Throughput: 2.2 FPS

Benchmarking Emotion Model...
  Avg inference time: 100.26ms
  Min: 82.95ms
  Max: 574.00ms
  Throughput: 10.0 FPS


Select mode:
1. Real-time webcam
2. Process video file

STARTING REAL-TIME RECOGNITION SYSTEM
Controls:
  - Press 'q' to quit
  - Press 's' to save current frame
  - Press 'p' to pause/resume

✅ System started. Processing frames...


⚠️ Interrupted by user

SESSION STATISTICS
Total frames processed: 56
Average FPS: 13.5
Average latency: 69.1ms

Select mode:
1. Real-time webcam
2. Process video file

📹 Processing video: C:\Users\Mohamed Montasser\Downloads\Video\New folder\v_HorseRace_g14_c04.avi
Video info: 320x240 @ 29 FPS, 285 frames
Processing...

Progress: 94.7% (270/285)
✅ Processed video saved: C:\Users\